In [1]:
"""
Step 3: Custom Dataset, DataLoaders, and EfficientNet-B2 model setup.
Run after v2_step2_clean_split.py has generated train_v2.csv, val_v2.csv, test_v2.csv
"""
 
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
from pathlib import Path

In [2]:
# ---- CONFIG ----
CODE_DIR = Path(r"D:\B.Tech\CE-AI\Sem 7\MP\Code work")
IMG_SIZE = 260  # native input size for EfficientNet-B2
BATCH_SIZE = 16  # safer for B2 on 6GB VRAM
NUM_WORKERS = 0
 
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
 
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
 
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [3]:
class BreedDataset(Dataset):
    def __init__(self, csv_path, class_to_idx, transform=None):
        self.df = pd.read_csv(csv_path)
        self.class_to_idx = class_to_idx
        self.transform = transform
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        label = self.class_to_idx[row["label"]]
        if self.transform:
            image = self.transform(image)
        return image, label

In [4]:
if __name__ == "__main__":
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {DEVICE}")
    if DEVICE.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
 
    # ---- Load class list ----
    with open(CODE_DIR / "classes_v2.txt", "r") as f:
        classes = [line.strip() for line in f.readlines()]
    NUM_CLASSES = len(classes)
    class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
    print(f"Number of classes: {NUM_CLASSES}")
 
    # ---- Build datasets ----
    train_dataset = BreedDataset(CODE_DIR / "train_v2.csv", class_to_idx, transform=train_transform)
    val_dataset = BreedDataset(CODE_DIR / "val_v2.csv", class_to_idx, transform=eval_transform)
    test_dataset = BreedDataset(CODE_DIR / "test_v2.csv", class_to_idx, transform=eval_transform)
 
    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples: {len(val_dataset)}")
    print(f"Test samples: {len(test_dataset)}")
 
    # ---- Build DataLoaders ----
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
 
    # ---- Sanity check: fetch one batch ----
    images, labels = next(iter(train_loader))
    print(f"\nSample batch -> images: {images.shape}, labels: {labels.shape}")
 
    # ---- Build EfficientNet-B2 model (transfer learning) ----
    model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
 
    # Replace the final classifier layer for our 62 classes
    in_features = model.classifier[1].in_features
    model.classifier[1] = torch.nn.Linear(in_features, NUM_CLASSES)
 
    model = model.to(DEVICE)
    print(f"\nModel loaded: EfficientNet-B2, final layer -> {NUM_CLASSES} classes")
 
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params: {total_params:,} | Trainable params: {trainable_params:,}")
 
    # ---- Quick GPU forward pass test (confirms no immediate OOM) ----
    model.eval()
    with torch.no_grad():
        test_out = model(images.to(DEVICE))
    print(f"Forward pass output shape: {test_out.shape}  (expected: [{BATCH_SIZE}, {NUM_CLASSES}])")
    print("GPU forward pass successful - no OOM at this batch size.")

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Number of classes: 62
Train samples: 9752
Val samples: 2090
Test samples: 2090

Sample batch -> images: torch.Size([16, 3, 260, 260]), labels: torch.Size([16])


Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to C:\Users\Mahipal/.cache\torch\hub\checkpoints\efficientnet_b2_rwightman-c35c1473.pth
100%|█████████████████████████████████████████████████████████████████████████████| 35.2M/35.2M [00:15<00:00, 2.41MB/s]



Model loaded: EfficientNet-B2, final layer -> 62 classes
Total params: 7,788,352 | Trainable params: 7,788,352
Forward pass output shape: torch.Size([16, 62])  (expected: [16, 62])
GPU forward pass successful - no OOM at this batch size.
